In [7]:
# %%
import logging
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Project root = thư mục cha của notebooks/
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "config").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "config").exists():
    raise RuntimeError("Could not locate project root from current working directory")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Project root: {PROJECT_ROOT}")
print(f"✅ Python: {sys.version.split()[0]}")

# Set DEBUG để thấy mọi log từ src/ modules
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
# Tắt noise từ thư viện bên ngoài
for _lib in ["matplotlib", "PIL", "lightgbm", "mlflow", "urllib3"]:
    logging.getLogger(_lib).setLevel(logging.WARNING)

warnings.filterwarnings("ignore", category=UserWarning)
print("✅ Logging ready (DEBUG)")

✅ Project root: C:\Users\ADMIN\Documents\project_churn_prediction
✅ Python: 3.11.9
✅ Logging ready (DEBUG)


In [8]:
# %%
from src.utils.config import load_config

CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
config = load_config(CONFIG_PATH)

print(f"📄 Config: {CONFIG_PATH}")
print(f"   random_state    : {config['project']['random_state']}")
print(f"   target_col      : {config['project']['target_col']}")
print(f"   candidates      : {config['modeling']['candidate_models']}")
print(f"   champion_metric : {config['modeling']['champion_metric']}")
print(f"   threshold       : {config['modeling']['decision_threshold']}")

02:16:52 | INFO     | src.utils.config | Config loaded from: C:\Users\ADMIN\Documents\project_churn_prediction\config\config.yaml
📄 Config: C:\Users\ADMIN\Documents\project_churn_prediction\config\config.yaml
   random_state    : 42
   target_col      : is_churn
   candidates      : ['logistic_regression', 'random_forest', 'lightgbm']
   champion_metric : auc_pr
   threshold       : 0.5


In [9]:
# %%
from sklearn.metrics import (
    average_precision_score, f1_score,
    precision_score, recall_score, roc_auc_score,
)

THRESHOLD = float(config["modeling"]["decision_threshold"])

def compute_metrics(y_true, y_score, threshold=THRESHOLD) -> dict:
    y_pred  = (y_score >= threshold).astype(int)
    n_top   = max(1, int(len(y_true) * 0.10))
    top_idx = np.argsort(y_score)[::-1][:n_top]
    lift_10 = float(np.array(y_true)[top_idx].mean()) / float(y_true.mean())
    return {
        "AUC-ROC":   round(float(roc_auc_score(y_true, y_score)), 4),
        "AUC-PR":    round(float(average_precision_score(y_true, y_score)), 4),
        "F1":        round(float(f1_score(y_true, y_pred, zero_division=0)), 4),
        "Precision": round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
        "Recall":    round(float(recall_score(y_true, y_pred, zero_division=0)), 4),
        "Lift@10%":  round(lift_10, 2),
    }

def print_metrics(name: str, train_m: dict, val_m: dict):
    print(f"\n{'─'*58}")
    print(f"  📈  {name}")
    print(f"{'─'*58}")
    print(f"  {'Metric':<14} {'Train':>10} {'Val':>10}  {'Gap':>8}")
    print(f"  {'─'*14} {'─'*10} {'─'*10}  {'─'*8}")
    for k in train_m:
        gap  = train_m[k] - val_m[k]
        flag = " ⚠️" if abs(gap) > 0.05 and k in ("AUC-ROC","AUC-PR") else ""
        print(f"  {k:<14} {train_m[k]:>10.4f} {val_m[k]:>10.4f}"
              f"  {gap:>+8.4f}{flag}")
    print(f"{'─'*58}\n")

print(f"✅ Metric helpers ready  |  threshold = {THRESHOLD}")

✅ Metric helpers ready  |  threshold = 0.5


In [10]:
# %%
# Load persisted splits and preprocessor (produced by src/features/run_preprocessing.py)
from pathlib import Path
from src.features.preprocess import load_splits, load_preprocessor

processed_dir = PROJECT_ROOT / "data" / "processed"
preprocessor_path = PROJECT_ROOT / "models" / "preprocessor.pkl"

try:
    splits = load_splits(processed_dir)
    X_train = splits.X_train
    X_val = splits.X_val
    X_test = splits.X_test
    y_train = splits.y_train
    y_val = splits.y_val
    y_test = splits.y_test
    preprocessor = load_preprocessor(preprocessor_path)
    print(f"Loaded splits from {processed_dir} and preprocessor from {preprocessor_path}")
except FileNotFoundError as err:
    raise RuntimeError(f"Missing preprocessing artifacts: {err}\nRun src/features/run_preprocessing.py first")


02:16:52 | INFO     | src.features.preprocess | Loaded X_train from C:\Users\ADMIN\Documents\project_churn_prediction\data\processed\X_train.parquet  (shape=(695051, 54))
02:16:52 | INFO     | src.features.preprocess | Loaded X_val from C:\Users\ADMIN\Documents\project_churn_prediction\data\processed\X_val.parquet  (shape=(148940, 54))
02:16:52 | INFO     | src.features.preprocess | Loaded X_test from C:\Users\ADMIN\Documents\project_churn_prediction\data\processed\X_test.parquet  (shape=(148940, 54))
02:16:52 | INFO     | src.features.preprocess | Loaded y_train from C:\Users\ADMIN\Documents\project_churn_prediction\data\processed\y_train.parquet  (shape=(695051, 1))
02:16:52 | INFO     | src.features.preprocess | Loaded y_val from C:\Users\ADMIN\Documents\project_churn_prediction\data\processed\y_val.parquet  (shape=(148940, 1))
02:16:53 | INFO     | src.features.preprocess | Loaded y_test from C:\Users\ADMIN\Documents\project_churn_prediction\data\processed\y_test.parquet  (shape=(1

In [11]:
# %%
# Build X riêng cho LR (OneHot) và Tree models (Ordinal đã có)
# ── Bước này chạy sau Cell 2 ──────────────────────────────────────────────
from src.features.preprocess import (
    identify_column_groups,
    build_linear_preprocessor,
    _sanitise_for_sklearn,
)

# Reload raw (un-transformed) splits để re-encode cho LR
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
raw_frame = pd.read_parquet(INTERIM_DIR / "modeling_frame.parquet")

# Lấy lại splits theo index
train_idx = X_train.index
val_idx   = X_val.index
test_idx  = X_test.index

target_col = config["project"]["target_col"]
raw_train = raw_frame.loc[train_idx]
raw_val   = raw_frame.loc[val_idx]
raw_test  = raw_frame.loc[test_idx]

# Identify columns
extra_drop = [c for c in ["bd", "gender", "city", "registered_via"]
              if c in raw_train.columns]
groups = identify_column_groups(
    raw_train.drop(columns=[target_col]), extra_drop=extra_drop
)

# Build + fit linear preprocessor (OneHotEncoder) trên train only
linear_prep = build_linear_preprocessor(groups)
X_train_raw_clean = _sanitise_for_sklearn(raw_train.drop(columns=[target_col]))
linear_prep.fit(X_train_raw_clean)

# Transform tất cả splits
def _transform_linear(df):
    arr = linear_prep.transform(_sanitise_for_sklearn(df.drop(columns=[target_col])))
    try:
        cols = linear_prep.get_feature_names_out()
    except Exception:
        cols = [f"f{i}" for i in range(arr.shape[1])]
    return pd.DataFrame(arr, index=df.index, columns=cols)

X_train_lr = _transform_linear(raw_train)
X_val_lr   = _transform_linear(raw_val)
X_test_lr  = _transform_linear(raw_test)

print(f"✅ LR feature matrix | train={X_train_lr.shape}, val={X_val_lr.shape}")
print(f"   (OneHotEncoder tạo {X_train_lr.shape[1]} features từ {X_train.shape[1]} ordinal features)")

02:16:53 | INFO     | src.features.preprocess | Column groups identified | numeric=24, categorical=0, drop=9
02:16:53 | INFO     | src.features.preprocess | Linear preprocessor built | SimpleImputer+RobustScaler on 24 numeric, SimpleImputer+OneHotEncoder on 0 categorical.
✅ LR feature matrix | train=(695051, 24), val=(148940, 24)
   (OneHotEncoder tạo 24 features từ 54 ordinal features)


In [16]:
# %%
from sklearn.linear_model import LogisticRegression

print("🚀 Training Logistic Regression (baseline)...")

lr_params = {
    "solver":       config["logistic_regression"]["solver"],
    "max_iter":     int(config["logistic_regression"]["max_iter"]),
    "class_weight": config["logistic_regression"]["class_weight"],
    "random_state": int(config["project"]["random_state"]),
    "n_jobs":       -1,
    "verbose":      1,
}
print(f"   params: {lr_params}\n")

t0 = time.perf_counter()
lr_model = LogisticRegression(**lr_params)

lr_model.fit(X_train_lr, y_train)          # ← đổi X_train → X_train_lr
lr_time = time.perf_counter() - t0

print(f"\n⏱  Done in {lr_time:.1f}s  "
      f"| n_iter={lr_model.n_iter_[0]}/{lr_params['max_iter']}")

lr_train_m = compute_metrics(y_train, lr_model.predict_proba(X_train_lr)[:, 1])  # ← X_train_lr
lr_val_m   = compute_metrics(y_val,   lr_model.predict_proba(X_val_lr)[:, 1])    # ← X_val_lr
print_metrics("Logistic Regression (OneHot — Fixed)", lr_train_m, lr_val_m)

🚀 Training Logistic Regression (baseline)...
   params: {'solver': 'lbfgs', 'max_iter': 1000, 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1, 'verbose': 1}



c:\Users\ADMIN\Documents\project_churn_prediction\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



⏱  Done in 1.2s  | n_iter=52/1000

──────────────────────────────────────────────────────────
  📈  Logistic Regression (OneHot — Fixed)
──────────────────────────────────────────────────────────
  Metric              Train        Val       Gap
  ────────────── ────────── ──────────  ────────
  AUC-ROC            0.7210     0.7166   +0.0044
  AUC-PR             0.2338     0.2284   +0.0054
  F1                 0.2138     0.2117   +0.0021
  Precision          0.1255     0.1242   +0.0013
  Recall             0.7210     0.7149   +0.0061
  Lift@10%           3.0100     2.9400   +0.0700
──────────────────────────────────────────────────────────



In [13]:
# %%
# Kiểm tra xem X_train_lr đã được tạo chưa
try:
    print(f"✅ X_train_lr exists | shape={X_train_lr.shape}")
    print(f"   NaN count: {X_train_lr.isna().sum().sum()}")
    print(f"   dtype sample: {X_train_lr.dtypes.value_counts().to_dict()}")
except NameError:
    print("❌ X_train_lr chưa tồn tại → cell tạo X_train_lr bị lỗi/chưa chạy")

# Kiểm tra LR đang dùng X nào
import inspect
# Nếu vẫn dùng X_train (ordinal) thì metrics sẽ vẫn sai
print(f"\nX_train shape   : {X_train.shape}")

✅ X_train_lr exists | shape=(695051, 24)
   NaN count: 0
   dtype sample: {dtype('float64'): 24}

X_train shape   : (695051, 54)


In [14]:
# %%
import os

print("📂 Files in data/processed/:")
for f in sorted((PROJECT_ROOT / "data" / "processed").iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"   {f.name:<40} {size_mb:.1f} MB")

print("\n📂 Files in data/interim/:")
for f in sorted((PROJECT_ROOT / "data" / "interim").iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"   {f.name:<40} {size_mb:.1f} MB")

📂 Files in data/processed/:
   feature_frame.parquet                    157.8 MB
   feature_metadata.json                    0.0 MB
   stratified_split_manifest.json           0.0 MB
   time_split_manifest.json                 0.0 MB
   X_test.parquet                           20.6 MB
   X_train.parquet                          84.3 MB
   X_val.parquet                            20.6 MB
   y_test.parquet                           1.0 MB
   y_train.parquet                          3.6 MB
   y_val.parquet                            1.0 MB

📂 Files in data/interim/:
   members.parquet                          309.2 MB
   modeling_frame.parquet                   104.3 MB
   train.parquet                            43.3 MB
   transactions_summary.parquet             126.4 MB
   user_logs_summary.parquet                428.0 MB
